# Editor de proyecto (anywidget)

Comprobación manual del widget: lo que no pueden verificar los tests de Python.

**Antes de empezar**: selecciona el kernel del `.venv` del proyecto (en VS Code,
arriba a la derecha → *Select Kernel* → *Python Environments* → `.venv`).

La interfaz es maestro-detalle: a la izquierda los componentes agrupados por tipo,
a la derecha el formulario del que tengas seleccionado. Los campos se generan del
esquema, así que cada uno sale con su unidad, sus límites y sus opciones.

La primera vez, el navegador descarga `ajv` de esm.sh: hace falta conexión.

## 1. Proyecto pequeño

In [1]:
import opensimula as osm

sim = osm.Simulation()
sim.console_print = False
pro = sim.new_project("proyecto")
pro.read_json("../test/test_project_1.json")

editor = pro.editor()
editor

### Qué mirar

1. Que aparezcan los dos paneles, no una caja vacía ni `Error displaying widget`.
2. Pincha un `Material`. El formulario debe traer `conductivity` como campo
   numérico con la unidad `W/(m·K)` al lado, y `use_resistance` como casilla.
3. Pon `conductivity` a `-1`: el campo se marca en rojo, con
   `must be >= 0` debajo y en la barra de estado. Si dijera *"must NOT have
   additional properties"*, el discriminador de Ajv no estaría actuando.
4. Mira un `Building_surface`: `construction` tiene que ser un **desplegable con
   los nombres de las Construction del proyecto**, no un campo de texto libre.
5. Déjalo válido antes de seguir.

## 2. ¿Llegan los cambios al kernel?

In [5]:
# Cambia algo en el widget de arriba, espera medio segundo y ejecuta esto.
editor.value["components"][3]

{'type': 'Construction',
 'name': 'Construction_1',
 'description': 'Construction using layers of material',
 'solar_alpha': [0.8, 0.8],
 'lw_epsilon': [0.9, 0.9],
 'materials': ['Materia_1'],
 'thicknesses': []}

In [6]:
print("válido:", editor.is_valid())
for linea in editor.error_report():
    print(" ", linea)

válido: False
  /components/3/materials/0: no component named "Materia_1"


Si `editor.value` no refleja lo que escribiste, la sincronización JS → Python
no funciona (consola del navegador: *Developer: Toggle Developer Tools*).

## 3. ¿Llegan los cambios del kernel al widget?

In [7]:
# Reasignar, nunca mutar: traitlets detecta los cambios por identidad.
editor.value = {**editor.value, "description": "cambiado desde Python"}

El panel de proyecto debe mostrar el nuevo `description` sin perder lo demás.

## 4. Un edificio real

In [8]:
hulc = sim.new_project("hulc")
hulc.read_json("edificio_curso_hulc.json")
print("componentes:", len(hulc.component_list()))

editor_hulc = hulc.editor()
editor_hulc

componentes: 150


Aquí se ve si aguanta un edificio de verdad: 150 componentes en 15 grupos.

**Prueba el aviso de referencias colgadas**: renombra una `Construction` y mira la
barra de estado. Debe avisar en amarillo de los `Building_surface` que se quedan
apuntando a un nombre que ya no existe. Eso no lo detecta el esquema — que un
nombre exista es propiedad del documento, no del tipo — así que es una
comprobación aparte.

## 5. Marimo

El mismo widget, envuelto:

```python
import marimo as mo
import opensimula as osm

sim = osm.Simulation()
pro = sim.new_project("proyecto")
pro.read_json("test/test_project_1.json")

editor = mo.ui.anywidget(pro.editor())
editor
```

En otra celda, `editor.value["value"]` es reactivo.

## Desarrollo del JS

Para tocar `editor.js` sin reiniciar el kernel, arranca con `ANYWIDGET_HMR=1`.